# 🧹 Data Preprocessing — House Price Prediction
Notebook này thực hiện làm sạch dữ liệu và tách thành 2 tập: Chung cư & Nhà đất.

**Input:** `data/raw/raw_data.csv`  
**Output:** `data/processed/cleaned_chung_cu.csv`, `cleaned_nha_dat.csv`, `cleaned_full_web.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import os

os.makedirs('../data/processed', exist_ok=True)
df = pd.read_csv('../data/raw/raw_data.csv', low_memory=False)
print(f"Ban đầu: {len(df)} dòng × {len(df.columns)} cột")

Ban đầu: 11831 dòng × 33 cột


## Bước 1: Xóa rác, Dedup & Sửa lỗi Scraper & Parse giá

In [2]:
# 1. Bỏ cột rỗng 100%
df = df.drop(columns=['toilets'], errors='ignore')

# 2. Xóa dòng trùng lặp
df = df.drop_duplicates(subset='listing_id')
print(f"Sau dedup: {len(df)} dòng")

# 3. Sửa lỗi Scraper: Swap cột price <-> price_per_m2 nếu bị đảo
mask = df['price'].str.contains('triệu/m', na=False)
df.loc[mask, ['price', 'price_per_m2']] = df.loc[mask, ['price_per_m2', 'price']].values
print(f"Đã sửa {mask.sum()} dòng bị lỗi đảo cột price")

# 4. Parse diện tích TRƯỚC (parse_price cần dùng)
df['area_m2'] = df['area'].str.extract(r'([\d,.]+)')[0].str.replace(',', '.').astype(float)

# 5. Parse Giá
def parse_price(price_str, area_m2=None):
    if pd.isna(price_str): return None
    p = str(price_str).lower().strip()
    if 'thỏa thuận' in p or 'thỏa' in p: return None
    
    # Lỗi scraper: "4 nghìn" thực ra là "4 tỷ"
    if 'nghìn' in p:
        num = re.findall(r'[\d,.]+', p)
        if num: return float(num[0].replace(',', '.'))
    
    if 'triệu/m' in p and area_m2:
        num = re.findall(r'[\d,.]+', p)
        if num:
            unit_price = float(num[0].replace(',', '.'))
            return round(unit_price * area_m2 / 1000, 2)
    elif 'tỷ' in p:
        num = re.findall(r'[\d,.]+', p)
        if num: return float(num[0].replace(',', '.'))
    elif 'triệu' in p:
        num = re.findall(r'[\d,.]+', p)
        if num: return round(float(num[0].replace(',', '.')) / 1000, 3)
    return None

df['price_billion'] = df.apply(lambda row: parse_price(row['price'], row.get('area_m2')), axis=1)

before = len(df)
df = df.dropna(subset=['price_billion'])
print(f"Sau khi xóa dòng không có giá: {len(df)} dòng (mất {before - len(df)})")

Sau dedup: 11830 dòng
Đã sửa 0 dòng bị lỗi đảo cột price
Sau khi xóa dòng không có giá: 11830 dòng (mất 0)


## Bước 2: Phân loại BĐS và Tách Nhánh

In [3]:
is_chung_cu = df['property_type'].str.contains('chung cư|Chung cư', na=False)
is_nha_rieng = df['property_type'].str.contains('nhà riêng|Nhà riêng|biệt thự|Biệt thự', na=False)

unclassified = df[~is_chung_cu & ~is_nha_rieng]
print(f"Không phân loại được: {len(unclassified)} dòng")

df_chung_cu = df[is_chung_cu].copy()
df_nha_dat = df[is_nha_rieng].copy()
print(f"Chung cư: {len(df_chung_cu)} | Nhà đất: {len(df_nha_dat)}")

Không phân loại được: 0 dòng
Chung cư: 5452 | Nhà đất: 6379


## Bước 3: Làm sạch chung cho cả 2 nhánh

In [4]:
def clean_common_features(data):
    data = data.copy()
    
    # 1. Tỉnh/TP + Quận/Huyện
    address_parts = data['address'].fillna('').str.split(' - ')
    data['city'] = address_parts.str[0].str.strip().replace('', 'Không rõ').fillna('Không rõ')
    data['district'] = address_parts.str[1].str.strip().replace('', 'Không rõ').fillna('Không rõ')
    data['location_key'] = data['city'] + ' - ' + data['district']
    
    location_counts = data['location_key'].value_counts()
    rare_locations = location_counts[location_counts < 30].index
    print(f"Khu vực bị gom vào '<Tỉnh/TP> - Khác': {list(rare_locations)}")
    data.loc[data['location_key'].isin(rare_locations), 'district'] = 'Khác'
    data['location_key'] = data['city'] + ' - ' + data['district']
    
    # 2. Phòng ngủ (Fill NaN bằng median theo quận TRƯỚC khi lọc outlier)
    data['bedrooms_num'] = data['bedrooms'].str.extract(r'(\d+)')[0].astype(float)
    data['bedrooms_num'] = data.groupby('location_key')['bedrooms_num'].transform(
        lambda x: x.fillna(x.median())
    )
    data['bedrooms_num'] = data['bedrooms_num'].fillna(data['bedrooms_num'].median())
    
    # 3. Hướng nhà & Ban công
    data['direction'] = data['direction'].fillna('Không rõ')
    if 'balcony_direction' in data.columns:
        data['balcony_direction'] = data['balcony_direction'].fillna('Không rõ')
    
    # 4. Nội thất
    def standardize_furniture(val):
        if pd.isna(val): return 'Không rõ'
        val = str(val).lower()
        if any(x in val for x in ['full', 'đầy đủ', 'cao cấp']): return 'Đầy đủ'
        if any(x in val for x in ['cơ bản', 'nguyên bản', 'cđt']): return 'Cơ bản'
        if any(x in val for x in ['không', 'trống']): return 'Không nội thất'
        return 'Không rõ'
    data['furniture_std'] = data['furniture'].apply(standardize_furniture)
    
    # 5. Pháp lý
    def standardize_legal(val):
        if pd.isna(val): return 'Không rõ'
        val = str(val).lower()
        if any(x in val for x in ['sổ đỏ', 'sổ hồng', 'sẵn sổ', 'có sổ']): return 'Sổ đỏ/Sổ hồng'
        if any(x in val for x in ['hợp đồng', 'hđmb']): return 'HĐMB'
        if 'chờ' in val: return 'Đang chờ sổ'
        return 'Khác'
    data['legal_std'] = data['legal'].apply(standardize_legal)
    
    return data

df_chung_cu = clean_common_features(df_chung_cu)
df_nha_dat = clean_common_features(df_nha_dat)

Khu vực bị gom vào '<Tỉnh/TP> - Khác': ['Hồ Chí Minh - Gò Vấp', 'Hà Nội - Hai Bà Trưng', 'Hà Nội - Đống Đa', 'Hồ Chí Minh - Quận 3', 'Hà Nội - Thanh Trì', 'Hà Nội - Ba Đình', 'Đà Nẵng - Thanh Khê', 'Hà Nội - Thạch Thất', 'Hồ Chí Minh - Cần Giờ', 'Hà Nội - Đan Phượng', 'Hồ Chí Minh - Hóc Môn', 'Hà Nội - Thường Tín']
Khu vực bị gom vào '<Tỉnh/TP> - Khác': ['Hà Nội - Đan Phượng', 'Hồ Chí Minh - Quận 4', 'Hà Nội - Chương Mỹ', 'Hồ Chí Minh - Quận 5', 'Hà Nội - Hoàn Kiếm', 'Hà Nội - Thanh Oai', 'Hà Nội - Quốc Oai', 'Hà Nội - Sóc Sơn', 'Hà Nội - Mê Linh', 'Hồ Chí Minh - Củ Chi', 'Hà Nội - Thạch Thất', 'Hồ Chí Minh - Cần Giờ', 'Hà Nội - Mỹ Đức', 'Hà Nội - Sơn Tây']


## Bước 4: Lọc Outlier cho Chung Cư

In [5]:
df_chung_cu = df_chung_cu.drop(columns=['floors', 'frontage', 'road_width'], errors='ignore')
df_chung_cu['project_name'] = df_chung_cu['project_name'].fillna('Không rõ')

before = len(df_chung_cu)
df_chung_cu = df_chung_cu[
    (df_chung_cu['area_m2'] >= 20) & (df_chung_cu['area_m2'] <= 300) &
    (df_chung_cu['price_billion'] >= 0.3) & (df_chung_cu['price_billion'] <= 50) &
    (df_chung_cu['bedrooms_num'] >= 1) & (df_chung_cu['bedrooms_num'] <= 6)
]
print(f"Chung cư sau outlier: {len(df_chung_cu)} (loại {before - len(df_chung_cu)})")

Chung cư sau outlier: 5452 (loại 0)


## Bước 5: Fill NA & Lọc Outlier cho Nhà Đất

In [6]:
df_nha_dat['floors_num'] = df_nha_dat['floors'].str.extract(r'(\d+)')[0].astype(float)
df_nha_dat['frontage_m'] = df_nha_dat['frontage'].str.extract(r'([\d,.]+)')[0].str.replace(',', '.').astype(float)
df_nha_dat['road_width_m'] = df_nha_dat['road_width'].str.extract(r'([\d,.]+)')[0].str.replace(',', '.').astype(float)

for col in ['floors_num', 'frontage_m', 'road_width_m']:
    df_nha_dat[col] = df_nha_dat.groupby('location_key')[col].transform(lambda x: x.fillna(x.median()))
    df_nha_dat[col] = df_nha_dat[col].fillna(df_nha_dat[col].median())

before = len(df_nha_dat)
df_nha_dat = df_nha_dat[
    (df_nha_dat['area_m2'] >= 15) & (df_nha_dat['area_m2'] <= 1000) &
    (df_nha_dat['price_billion'] >= 0.5) & (df_nha_dat['price_billion'] <= 200) &
    (df_nha_dat['bedrooms_num'] >= 1) & (df_nha_dat['bedrooms_num'] <= 25) &
    (df_nha_dat['road_width_m'] <= 50)
]
print(f"Nhà đất sau outlier: {len(df_nha_dat)} (loại {before - len(df_nha_dat)})")

Nhà đất sau outlier: 6379 (loại 0)


## Bước 6: Xuất file

In [7]:
features_cc = ['price_billion', 'area_m2', 'bedrooms_num', 'city', 'district', 'location_key', 'direction', 'balcony_direction', 'furniture_std', 'legal_std', 'title', 'description', 'image_urls']
df_chung_cu[features_cc].to_csv('../data/processed/cleaned_chung_cu.csv', index=False)

features_nd = ['price_billion', 'area_m2', 'bedrooms_num', 'city', 'district', 'location_key', 'direction', 'furniture_std', 'legal_std', 'floors_num', 'frontage_m', 'road_width_m', 'title', 'description', 'image_urls']
df_nha_dat[features_nd].to_csv('../data/processed/cleaned_nha_dat.csv', index=False)

df_full = pd.concat([df_chung_cu, df_nha_dat])
df_full.to_csv('../data/processed/cleaned_full_web.csv', index=False)

print(f"✅ Chung cư: {len(df_chung_cu)} dòng × {len(features_cc)} features")
print(f"✅ Nhà đất: {len(df_nha_dat)} dòng × {len(features_nd)} features")
print(f"✅ Full web: {len(df_full)} dòng")

✅ Chung cư: 5452 dòng × 13 features
✅ Nhà đất: 6379 dòng × 15 features
✅ Full web: 11831 dòng


## Kiểm tra nhanh sau xuất

In [8]:
# Verify: không còn NaN trong features
cc = pd.read_csv('../data/processed/cleaned_chung_cu.csv')
nd = pd.read_csv('../data/processed/cleaned_nha_dat.csv')
print('=== NaN check Chung cư ===')
print(cc.isnull().sum())
print(f'\n=== NaN check Nhà đất ===')
print(nd.isnull().sum())
print(f'\n=== Describe Chung cư ===')
print(cc.describe())
print(f'\n=== Describe Nhà đất ===')
print(nd.describe())

=== NaN check Chung cư ===
price_billion        0
area_m2              0
bedrooms_num         0
city                 0
district             0
location_key         0
direction            0
balcony_direction    0
furniture_std        0
legal_std            0
title                0
description          0
image_urls           0
dtype: int64

=== NaN check Nhà đất ===
price_billion    0
area_m2          0
bedrooms_num     0
city             0
district         0
location_key     0
direction        0
furniture_std    0
legal_std        0
floors_num       0
frontage_m       0
road_width_m     0
title            0
description      0
image_urls       0
dtype: int64

=== Describe Chung cư ===
       price_billion      area_m2  bedrooms_num
count    5452.000000  5452.000000   5452.000000
mean        7.350016    80.605596      2.170763
std         6.021124    33.664321      0.702750
min         0.693000    20.000000      1.000000
25%         3.650000    60.000000      2.000000
50%         5.500000 

In [9]:
# Hiển thị 5 dòng đầu tiên dưới dạng bảng
display(cc.head())
display(nd.head())

,price_billion,area_m2,bedrooms_num,city,district,location_key,direction,balcony_direction,furniture_std,legal_std,title,description,image_urls
0,6.40,75.0,2.0,Hà Nội,Hoài Đức,Hà Nội - Hoài Đức,Không rõ,Không rõ,Cơ bản,HĐMB,Vành Đai 3.5 + Metro số 7: Công thức nhân đôi ...,Siêu phẩm căn 2 ngủ tại The Flame Vine - Hinod...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
1,19.20,118.2,3.0,Hà Nội,Tây Hồ,Hà Nội - Tây Hồ,Đông - Nam,Tây - Bắc,Cơ bản,HĐMB,"BÁN 2CĂN 3PN DỰ ÁN CELESTINE WESTLAKE, VIEW HỒ...",Dự án CELESTINE WESTLAKE số 300 Võ Chí Công - ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
2,7.55,75.0,2.0,Hà Nội,Hoàng Mai,Hà Nội - Hoàng Mai,Đông - Nam,Tây - Bắc,Đầy đủ,Sổ đỏ/Sổ hồng,"Chính chủ-Bán gấp chung cư tòa The Zen Gamuda,...",Chính chủ bán gấp căn hộ chung cư 2PN tòa The ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
3,9.20,99.1,3.0,Hà Nội,Tây Hồ,Hà Nội - Tây Hồ,Không rõ,Đông - Nam,Đầy đủ,Sổ đỏ/Sổ hồng,Bán gấp căn góc 3 p ngủ - 99.1m2 (thông thuỷ) ...,Chủ nhà cần bán gấp đưa giá nhanh vào việc:\nB...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
4,9.30,97.5,3.0,Hà Nội,Bắc Từ Liêm,Hà Nội - Bắc Từ Liêm,Đông - Nam,Bắc,Đầy đủ,Sổ đỏ/Sổ hồng,"Gấp bán căn góc 3 phòng ngủ - 97,5 m2 thông th...","Bán căn hộ 3 ngủ Sunshine City - Giá: 9,3 tỷ (...","[""https://file4.batdongsan.com.vn/resize/1275x..."


,price_billion,area_m2,bedrooms_num,city,district,location_key,direction,furniture_std,legal_std,floors_num,frontage_m,road_width_m,title,description,image_urls
0,9.9,98.0,3.0,Hồ Chí Minh,Khác,Hồ Chí Minh - Khác,Không rõ,Cơ bản,Sổ đỏ/Sổ hồng,4.0,4.00,5.0,"Q4, TÔN THẤT THUYẾT-GIÁ: 9,9TY - NGANG 7M, DIỆ...",NHÀ HẺM QUẬN 4 DIỆN TÍCH LỚN 98 M2 (CÔNG NHẬN...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
1,15.5,47.7,3.0,Hà Nội,Đống Đa,Hà Nội - Đống Đa,Không rõ,Không rõ,Không rõ,3.0,4.25,3.5,CBNN bán nhà sổ đỏ chính chủ trung tâm Nguyễn ...,Vị trí VIP bám đường rộng 26m theo QH 1/500 qu...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
2,11.6,72.0,2.0,Hà Nội,Gia Lâm,Hà Nội - Gia Lâm,Không rõ,Không rõ,Không rõ,2.0,6.00,6.0,"BÁN NHÀ PHÂN LÔ, KINH DOANH PHỐ TRÂU QUỲ GIA L...","+ Nhà đang cho thuê kinh doanh cafe, một tháng...","[""https://file4.batdongsan.com.vn/resize/1275x..."
3,15.8,46.0,4.0,Hà Nội,Cầu Giấy,Hà Nội - Cầu Giấy,Không rõ,Không rõ,Sổ đỏ/Sổ hồng,6.0,4.70,6.0,"Nhà Nguyễn Văn Huyên 46m2*6T xây mới, thang má...","Nhà mới đẹp ở ngay, full nội thất chỉ việc xác...","[""https://file4.batdongsan.com.vn/resize/1275x..."
4,8.0,40.0,6.0,Hà Nội,Nam Từ Liêm,Hà Nội - Nam Từ Liêm,Không rõ,Đầy đủ,Sổ đỏ/Sổ hồng,5.0,4.40,3.0,"Tôi chủ nhà bán nhà 6 phòng ngủ, 40m2 5T tại P...",Tôi chính chủ trên sổ đỏ bán nhà Lê Quang Đạo ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."


In [10]:
# Hiển thị 5 dòng đầu tiên dưới dạng bảng
display(cc.head())
display(nd.head())

,price_billion,area_m2,bedrooms_num,city,district,location_key,direction,balcony_direction,furniture_std,legal_std,title,description,image_urls
0,6.40,75.0,2.0,Hà Nội,Hoài Đức,Hà Nội - Hoài Đức,Không rõ,Không rõ,Cơ bản,HĐMB,Vành Đai 3.5 + Metro số 7: Công thức nhân đôi ...,Siêu phẩm căn 2 ngủ tại The Flame Vine - Hinod...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
1,19.20,118.2,3.0,Hà Nội,Tây Hồ,Hà Nội - Tây Hồ,Đông - Nam,Tây - Bắc,Cơ bản,HĐMB,"BÁN 2CĂN 3PN DỰ ÁN CELESTINE WESTLAKE, VIEW HỒ...",Dự án CELESTINE WESTLAKE số 300 Võ Chí Công - ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
2,7.55,75.0,2.0,Hà Nội,Hoàng Mai,Hà Nội - Hoàng Mai,Đông - Nam,Tây - Bắc,Đầy đủ,Sổ đỏ/Sổ hồng,"Chính chủ-Bán gấp chung cư tòa The Zen Gamuda,...",Chính chủ bán gấp căn hộ chung cư 2PN tòa The ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
3,9.20,99.1,3.0,Hà Nội,Tây Hồ,Hà Nội - Tây Hồ,Không rõ,Đông - Nam,Đầy đủ,Sổ đỏ/Sổ hồng,Bán gấp căn góc 3 p ngủ - 99.1m2 (thông thuỷ) ...,Chủ nhà cần bán gấp đưa giá nhanh vào việc:\nB...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
4,9.30,97.5,3.0,Hà Nội,Bắc Từ Liêm,Hà Nội - Bắc Từ Liêm,Đông - Nam,Bắc,Đầy đủ,Sổ đỏ/Sổ hồng,"Gấp bán căn góc 3 phòng ngủ - 97,5 m2 thông th...","Bán căn hộ 3 ngủ Sunshine City - Giá: 9,3 tỷ (...","[""https://file4.batdongsan.com.vn/resize/1275x..."


,price_billion,area_m2,bedrooms_num,city,district,location_key,direction,furniture_std,legal_std,floors_num,frontage_m,road_width_m,title,description,image_urls
0,9.9,98.0,3.0,Hồ Chí Minh,Khác,Hồ Chí Minh - Khác,Không rõ,Cơ bản,Sổ đỏ/Sổ hồng,4.0,4.00,5.0,"Q4, TÔN THẤT THUYẾT-GIÁ: 9,9TY - NGANG 7M, DIỆ...",NHÀ HẺM QUẬN 4 DIỆN TÍCH LỚN 98 M2 (CÔNG NHẬN...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
1,15.5,47.7,3.0,Hà Nội,Đống Đa,Hà Nội - Đống Đa,Không rõ,Không rõ,Không rõ,3.0,4.25,3.5,CBNN bán nhà sổ đỏ chính chủ trung tâm Nguyễn ...,Vị trí VIP bám đường rộng 26m theo QH 1/500 qu...,"[""https://file4.batdongsan.com.vn/resize/1275x..."
2,11.6,72.0,2.0,Hà Nội,Gia Lâm,Hà Nội - Gia Lâm,Không rõ,Không rõ,Không rõ,2.0,6.00,6.0,"BÁN NHÀ PHÂN LÔ, KINH DOANH PHỐ TRÂU QUỲ GIA L...","+ Nhà đang cho thuê kinh doanh cafe, một tháng...","[""https://file4.batdongsan.com.vn/resize/1275x..."
3,15.8,46.0,4.0,Hà Nội,Cầu Giấy,Hà Nội - Cầu Giấy,Không rõ,Không rõ,Sổ đỏ/Sổ hồng,6.0,4.70,6.0,"Nhà Nguyễn Văn Huyên 46m2*6T xây mới, thang má...","Nhà mới đẹp ở ngay, full nội thất chỉ việc xác...","[""https://file4.batdongsan.com.vn/resize/1275x..."
4,8.0,40.0,6.0,Hà Nội,Nam Từ Liêm,Hà Nội - Nam Từ Liêm,Không rõ,Đầy đủ,Sổ đỏ/Sổ hồng,5.0,4.40,3.0,"Tôi chủ nhà bán nhà 6 phòng ngủ, 40m2 5T tại P...",Tôi chính chủ trên sổ đỏ bán nhà Lê Quang Đạo ...,"[""https://file4.batdongsan.com.vn/resize/1275x..."


Bad pipe message: %s [b'\x12L/\x86\x04\x0bC\x9eLAXa\xf7\x91\xafT9\x11']
Bad pipe message: %s [b'1\r\nContent-Type: application/json\r\nContent-Lengt']
Bad pipe message: %s [b'\x1d\x7f\xb37\xadZ\xb4\xe4\n\x06\xe5\x8f\xa5"\xf3\xd8\x9ac\xed\xb7d\xb5j\x8e\x9e\x180\x8dM\xadw']
Bad pipe message: %s [b' 2\r\nx-codeium-csrf-token: 70826623-8bae-4f46-bec9-01f25b0d3ffb\r\nConnect-Protocol-Version: 1\r\nHost: 127.']
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/tangoctai/Library/Python/3.9/lib/python/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/tangoctai/Library/Python/3.9/lib/python/site-packages/ipykernel/kernelbase.py", line 302, in dispatch_control
    await self.process_control(msg)
  File "/Users/tangoctai/Library/Python/3.9/lib/python/site-packages/ipykernel/kernelbase.py", line 308, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
  File 